# Spatial Voting Analysis with socialchoicelab

This notebook walks through a typical spatial voting analysis using the
`socialchoicelab` Python package.

## Setup (first time, or after a fresh clone)

**1. Build the C library** (from the repo root in a terminal):
```bash
cmake -S . -B build && cmake --build build
```

**2. Install the Python package** in editable mode (from the `python/` directory):
```bash
pip install -e ".[dev]"
```

**3. Set the library path** so Python can find `libscs_api` at runtime.
Add this to your `~/.zshrc` (or run it in the terminal before launching Cursor/Jupyter):
```bash
export SCS_LIB_PATH=/path/to/socialchoicelab/build
```

**4. Select the correct kernel** in Cursor/JupyterLab — the Python environment
where you ran `pip install` above.

After setup, run the verification cell below. You should see the API version printed.
Full setup details: `python/README.md`.

In [ ]:
import numpy as np
import socialchoicelab as scl

print("C API version:", scl.scs_version())

## 1 — Voter ideal points in 2D

We model a legislature with **5 voters** in a two-dimensional issue space.

In [ ]:
# Flat [x0, y0, x1, y1, ...] representation.
voters = np.array([
    -1.0, -0.5,   # voter 0: left-economic, moderate-social
     0.0,  0.0,   # voter 1: centrist
     0.8,  0.6,   # voter 2: right-economic, conservative
    -0.4,  0.8,   # voter 3: centre-left, liberal
     0.5, -0.7,   # voter 4: right-economic, libertarian
])

# Three policy alternatives (0-based: 0=SQ, 1=A, 2=B)
alts = np.array([0.0, 0.0, 0.6, 0.4, -0.5, 0.3])  # SQ, A, B

sq = alts[:2]   # status quo
a  = alts[2:4]  # reform A
b  = alts[4:6]  # reform B

print("Status quo:", sq)
print("Reform A:  ", a)
print("Reform B:  ", b)

## 2 — Reproducible randomness with StreamManager

In [ ]:
sm = scl.StreamManager(master_seed=2024, stream_names=["gen", "ties"])
# Test draw
print("Sample uniform [0,1]:", sm.uniform_real("gen", 0.0, 1.0))

## 3 — Majority preference between two alternatives

In [ ]:
a_beats_sq = scl.majority_prefers_2d(a[0], a[1], sq[0], sq[1], voters)
b_beats_sq = scl.majority_prefers_2d(b[0], b[1], sq[0], sq[1], voters)

print("A beats SQ by majority:", a_beats_sq)
print("B beats SQ by majority:", b_beats_sq)

## 4 — Winset: which policies beat the status quo?

In [ ]:
ws = scl.winset_2d(alt_xy=alts[:2], voter_ideals_xy=voters)
print("Winset empty:", ws.is_empty())
print("Bounding box:", ws.bbox())

In [ ]:
try:
    import matplotlib.pyplot as plt
    from matplotlib.patches import Polygon
    from matplotlib.collections import PatchCollection

    bnd = ws.boundary()
    fig, ax = plt.subplots(figsize=(5, 5))
    if bnd is not None and len(bnd["xy"]) > 0:
        xy = bnd["xy"]
        ax.fill(xy[:, 0], xy[:, 1], alpha=0.3, color="steelblue", label="Winset")
        ax.plot(xy[:, 0], xy[:, 1], color="steelblue")

    voter_xy = voters.reshape(-1, 2)
    ax.scatter(voter_xy[:, 0], voter_xy[:, 1], marker="x", color="grey", label="Voters")
    for pt, label in [(sq, "SQ"), (a, "A"), (b, "B")]:
        ax.scatter(*pt, s=80, zorder=5)
        ax.annotate(label, pt, textcoords="offset points", xytext=(6, 4))

    ax.set_aspect("equal")
    ax.legend()
    ax.set_title("Winset of the status quo")
    plt.tight_layout()
    plt.show()
except ImportError:
    print("matplotlib not installed — skipping plot")

## 5 — Copeland winner (finite alternative set)

In [ ]:
scores = scl.copeland_scores_2d(alts, voters)
winner = scl.copeland_winner_2d(alts, voters)
print("Copeland scores:", scores)
print("Copeland winner (0-based):", winner)

## 6 — Condorcet winner

In [ ]:
has_cw = scl.has_condorcet_winner_2d(alts, voters)
cw     = scl.condorcet_winner_2d(alts, voters)
print("Has Condorcet winner:", has_cw)
print("Condorcet winner (0-based):", cw)  # None if not found

## 7 — Ordinal preference profiles and voting rules

In [ ]:
prof = scl.profile_build_spatial(alts, voters)
print("Dims:", prof.dims())       # (n_voters, n_alts)
print("Rankings:\n", prof.export())

In [ ]:
print("Plurality scores:", scl.plurality_scores(prof))
print("Plurality winner:", scl.plurality_one_winner(prof))

In [ ]:
print("Borda scores:  ", scl.borda_scores(prof))
print("Borda ranking: ", scl.borda_ranking(prof))  # best first

In [ ]:
# Approval voting: each voter approves their top 2
print("Top-2 approval scores:", scl.approval_scores_topk(prof, k=2))
print("Top-2 approval winners:", scl.approval_all_winners_topk(prof, k=2))

## 8 — Pareto efficiency and Condorcet consistency

In [ ]:
print("Pareto set:", scl.pareto_set(prof))           # 0-based indices
print("Has Condorcet winner (profile):", scl.has_condorcet_winner_profile(prof))
print("Condorcet winner (profile):",     scl.condorcet_winner_profile(prof))

for i in range(prof.dims()[1]):
    print(f"  alt {i}: Condorcet-consistent = {scl.is_condorcet_consistent(prof, i)},"
          f" majority-consistent = {scl.is_majority_consistent(prof, i)}")